In [ ]:
# install and download spaCy related modules
!pip install --upgrade spacy
!python -m spacy download en_core_web_lg

# spaCy
import spacy
from spacy.language import Language
from spacy.tokens import DocBin, Span
from spacy.matcher import PhraseMatcher
from spacy.kb import InMemoryLookupKB
from spacy.training import Example
from spacy.ml.models import load_kb
from spacy.util import minibatch, compounding

# Google Drive
from google.colab import drive

# Firebase/Firestore
import firebase_admin
from firebase_admin import credentials
from firebase_admin import firestore

# general Python modules
import json
import datetime
import requests
import csv
import random
from collections import Counter

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
/usr/local/lib/python3.9/dist-packages/torch/cuda/__init__.py:497: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")
2023-03-20 00:49:48.322011: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2023-03-20 00:49:49.430313: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libnvinfer.so.7'; dlerror: libnvinfer.so.7: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /usr/local/nvidia/lib:/usr/local/nvidia/lib64
2023-03-20 00:49:49.430405: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could no

/usr/local/lib/python3.9/dist-packages/torch/cuda/__init__.py:497: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


In [ ]:
from google.colab import drive

In [ ]:
# remount drive, forced if needed
drive.mount("/content/gdrive/", force_remount = True)
print("Stablished access to Google Drive")

# initialize Drive path
DRIVE_PATH = "/content/gdrive/My Drive"

Mounted at /content/gdrive/
Stablished access to Google Drive


In [ ]:
# input files
ents_file = DRIVE_PATH + "/ie_course_2022_team08/assets/entities.csv"
annot_text_file = DRIVE_PATH + "/ie_course_2022_team08/assets/annotations.json"
# output files
kb_dir = DRIVE_PATH + "/ie_course_2022_team08/output/ml_el/kb"
nlp_dir = DRIVE_PATH + "/ie_course_2022_team08/output/ml_el/my_nlp"
train_corpus = DRIVE_PATH + "/ie_course_2022_team08/output/ml_el/train_corpus"
test_corpus = DRIVE_PATH + "/ie_course_2022_team08/output/ml_el/test_corpus"
nlp_el_dir = DRIVE_PATH + "/ie_course_2022_team08/output/ml_el/my_el_nlp"
coref_dir = DRIVE_PATH + "/ie_course_2022_team08/output/ml_el/coref_nlp"

In [ ]:
# Helper function to read in the pre-defined entities we want to disambiguate to
def load_entities():
  names = dict()
  descriptions = dict()
  # read and iterate entities and split it into two dicts
  with open(ents_file, newline="") as f:
    entities = csv.reader(f, delimiter=",")
    for row in entities:
      qid = row[0]
      name = row[1]
      desc = row[2]
      names[qid] = name
      descriptions[qid] = desc
  return names, descriptions

In [ ]:
name_dict, desc_dict = load_entities()
print(name_dict)
print(desc_dict)

{'Q64705310': 'Brian Armstrong', 'Q1615': 'Neil Armstrong', 'Q2172': 'Lance Armstrong'}
{'Q64705310': 'American business executive and invester', 'Q1615': 'American astronaut and lunar explorer', 'Q2172': 'American cyclist'}


In [ ]:
# First: create a simple model with an NER component
# To ensure we get the correct entities for this demo, add a simple entity_ruler as well.
nlp = spacy.load("en_core_web_lg", exclude="parser, tagger, lemmatizer")
ruler = nlp.add_pipe("entity_ruler", after="ner")
patterns = [{"label": "PERSON", "pattern": [{"LOWER": "armstrong"}]}]
ruler.add_patterns(patterns)
nlp.add_pipe("sentencizer", first=True)

kb = InMemoryLookupKB(vocab=nlp.vocab, entity_vector_length=300)
for qid, desc in desc_dict.items():
    desc_doc = nlp(desc)
    desc_enc = desc_doc.vector
    kb.add_entity(entity=qid, entity_vector=desc_enc, freq=342)   # 342 is an arbitrary value here

for qid, name in name_dict.items():
  # set 100% prior probability P(entity|alias) for each unique name
  kb.add_alias(alias=name, entities=[qid], probabilities=[1])

qids = name_dict.keys()
probs = [0.3 for qid in qids]
# ensure that sum([probs]) <= 1 when setting aliases
kb.add_alias(alias="Armstrong", entities=qids, probabilities=probs)  #

print(f"Entities in the KB: {kb.get_entity_strings()}")
print(f"Aliases in the KB: {kb.get_alias_strings()}")
print()

# store knowledgebase and NLP pipeline
kb.to_disk(kb_dir)
print(f"Saved KB in: {kb_dir}")
nlp.to_disk(nlp_dir)
print(f"Saved NLP pipeline in: {nlp_dir}")

Entities in the KB: ['Q2172', 'Q1615', 'Q64705310']
Aliases in the KB: ['Lance Armstrong', 'Neil Armstrong', 'Armstrong', 'Brian Armstrong']

Saved KB in: /content/gdrive/My Drive/ie_course_2022_team08/output/ml_el/kb
Saved NLP pipeline in: /content/gdrive/My Drive/ie_course_2022_team08/output/ml_el/my_nlp


In [ ]:
nlp = spacy.load(nlp_dir, exclude="parser, tagger")
docs = []
gold_ids = []

with open(annot_text_file,"r", encoding="utf8") as f:
  for line in f: 
    example = json.loads(line)
    # print(example)
    annotations = example["annotations"]
    # print(annotations)
    for annotation in annotations:
      # print(annotation[1]["entities"][0][1])
      sentence= annotation[0]
      QID= annotation[1]["entities"][0][2]
      # print(QID)
      doc = nlp.make_doc(sentence)
      gold_ids.append(QID)
      # we assume only 1 annotated span per sentence, and only 1 KB ID per span
      entity = doc.char_span(
        annotation[1]["entities"][0][0],
        annotation[1]["entities"][0][1],
        label="PERSON",
        kb_id=QID,
      )
      doc.ents = [entity]
      for i, t in enumerate(doc):
        doc[i].is_sent_start = i == 0
      docs.append(doc)

print("Statistics of manually annotated data:")
print(Counter(gold_ids))
print()

train_docs = DocBin()
test_docs = DocBin()
for QID in ["Q64705310", "Q1615", "Q2172"]:
  indices = [i for i, j in enumerate(gold_ids) if j == QID]
  # first 8 in training
  for index in indices[0:8]:
    train_docs.add(docs[index])
  # last 2 in test
  for index in indices[8:10]:
    test_docs.add(docs[index])

train_docs.to_disk(train_corpus)
print(f"Saved train corpus in: {train_corpus}")
test_docs.to_disk(test_corpus)
print(f"Saved test corpus in: {test_corpus}")

Statistics of manually annotated data:
Counter({'Q2172': 10, 'Q1615': 10, 'Q64705310': 10})

Saved train corpus in: /content/gdrive/My Drive/ie_course_2022_team08/output/ml_el/train_corpus
Saved test corpus in: /content/gdrive/My Drive/ie_course_2022_team08/output/ml_el/test_corpus


In [ ]:
""" Step 3: Train entity linking model. """

nlp = spacy.load(nlp_dir)

TRAIN_EXAMPLES = []

with open(test_corpus, "rb") as f:
  doc_bin = DocBin().from_disk(test_corpus)
  docs = doc_bin.get_docs(nlp.vocab)
  for doc in docs:
    TRAIN_EXAMPLES.append(Example(nlp(doc.text), doc))

entity_linker = nlp.add_pipe("entity_linker", config={"incl_prior": False}, last=True)
entity_linker.initialize(lambda: TRAIN_EXAMPLES, nlp=nlp, kb_loader=load_kb(kb_dir))

with nlp.select_pipes(enable=["entity_linker"]):  # train only the entity_linker
  optimizer = nlp.resume_training()
  for itn in range(500):  # 500 iterations takes about a minute to train
    random.shuffle(TRAIN_EXAMPLES)
    batches = minibatch(TRAIN_EXAMPLES, size=compounding(4.0, 32.0, 1.001))  # increasing batch sizes
    losses = {}
    for batch in batches:
      nlp.update(
        batch,
        drop=0.2,  # prevent overfitting
        losses=losses,
        sgd=optimizer,
      )
    if itn % 50 == 0:
      print(itn, "Losses", losses)  # print the training loss
print(itn, "Losses", losses)

nlp.to_disk(nlp_el_dir)
print()
print(f"Saved NLP pipeline in: {nlp_el_dir}")

0 Losses {'entity_linker': 1.921010434627533}
50 Losses {'entity_linker': 0.04393075406551361}
100 Losses {'entity_linker': 0.018502652645111084}
150 Losses {'entity_linker': 0.016279175877571106}
200 Losses {'entity_linker': 0.010079383850097656}
250 Losses {'entity_linker': 0.00873592495918274}
300 Losses {'entity_linker': 0.007017359137535095}
350 Losses {'entity_linker': 0.004851028323173523}
400 Losses {'entity_linker': 0.006144315004348755}
450 Losses {'entity_linker': 0.004596292972564697}
499 Losses {'entity_linker': 0.003133431077003479}

Saved NLP pipeline in: /content/gdrive/My Drive/ie_course_2022_team08/output/ml_el/my_el_nlp


In [ ]:
""" Step 4: Evaluate the new Entity Linking component by applying it to unseen text. """

nlp = spacy.load(nlp_el_dir)

examples = []

with open(test_corpus, "rb") as f:
  doc_bin = DocBin().from_disk(test_corpus)
  docs = doc_bin.get_docs(nlp.vocab)
  for doc in docs:
    examples.append(Example(nlp(doc.text), doc))


print("RESULTS ON THE DEV SET:")
print()

for example in examples:
  print(example.text)
  print(f"Gold annotation: {example.reference.ents[0].kb_id_}")
  print(f"Predicted annotation: {example.predicted.ents[0].kb_id_}")
  print()

print()
print("RUNNING THE PIPELINE ON UNSEEN TEXT:")
text = "Armstrong owns about 19% of Coinbase's shares."
doc = nlp(text)
print(text)
for ent in doc.ents:
  print(ent.text, ent.label_, ent.kb_id_)
print()

RESULTS ON THE DEV SET:

Armstrong wrote a blog post in September 2020 calling Coinbase a "Mission Focused Company", discouraging employee activism and discussion of political and social issues at work.
Gold annotation: Q64705310
Predicted annotation: Q64705310

Armstrong was its first CEO.
Gold annotation: Q64705310
Predicted annotation: Q64705310

Armstrong engaged the Reentry Control System (RCS) and turned off the OAMS.
Gold annotation: Q1615
Predicted annotation: Q1615

Armstrong and Scott received the NASA Exceptional Service Medal, and the Air Force awarded Scott the Distinguished Flying Cross as well.
Gold annotation: Q1615
Predicted annotation: Q1615

Armstrong denied all such allegations until January 2013, often claiming that he never had any positive test in the drug tests he has taken over his cycling career.
Gold annotation: Q2172
Predicted annotation: Q2172

Armstrong rode up alongside on the Alpe d'Huez stage to tell him "it was a mistake to speak out the way I (Bassons

In [ ]:
!pip install crosslingual_coreference

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 45.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 719.6/719.6 KB 60.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.0/6.0 MB 76.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 463.2/463.2 KB 48.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.7/134.7 KB 15.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.9/77.9 KB 10.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.5/78.5 KB 8.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.0/21.0 MB 9.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 305.9/305.9 KB 32.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 593.6/593.6 KB 53.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━

In [ ]:
# Adding coref component:
# import crosslingual_coreference
# nlp = spacy.load("en_core_web_sm")
nlp.pipe_names
nlp.add_pipe("xx_coref",config={"chunk_size":2500,"chunk_overlap":2,"device":-1})


In [ ]:
!pip install --upgrade spacy-experimental

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/


In [ ]:
import spacy
import spacy_experimental

/usr/local/lib/python3.8/dist-packages/torch/cuda/__init__.py:497: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


In [ ]:
!pip install https://github.com/explosion/spacy-experimental/releases/download/v0.6.1/en_coreference_web_trf-3.4.0a2-py3-none-any.whl#egg=en_coreference_

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 490.3/490.3 MB 2.6 MB/s eta 0:00:00
  Attempting uninstall: en-coreference-web-trf
    Found existing installation: en-coreference-web-trf 3.4.0a0
    Uninstalling en-coreference-web-trf-3.4.0a0:
      Successfully uninstalled en-coreference-web-trf-3.4.0a0


In [ ]:
!pip install --upgrade spacy
!pip install --upgrade spacy-experimental
!pip install https://github.com/explosion/spacy-experimental/releases/download/v0.6.1/en_coreference_web_trf-3.4.0a2-py3-none-any.whl#egg=en_coreference_
import spacy
import spacy_experimental
nlp = spacy.load("en_coreference_web_trf")

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 673.8/673.8 KB 9.5 MB/s eta 0:00:00
Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 490.3/490.3 MB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.5/6.5 MB 69.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.5/53.5 KB 7.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.8/5.8 MB 77.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 60.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.6/7.6 MB 72.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.8/199.8 KB 14.5 MB/s eta 0:00:00
  Attempting uninstall: wasabi
    Found existing installation: 

/usr/local/lib/python3.9/dist-packages/torch/cuda/__init__.py:497: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


In [ ]:
doc = nlp("Ripple in partnership with Polygon to improve their payment system is another exciting development in the rapidly evolving cryptocurrency market")

In [ ]:
doc.spans

{'coref_clusters_1': [Polygon, their]}